In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm  = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

#this chekpointer creates checkpoint at eevery node means we can remeber it and store to memeory
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
# this config adds thread value means we can assign thread val and access it later 
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)',
 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to emotions or a personal issue. However, the punchline "Because it was feeling a little crusty!" subverts this expectation by using a wordplay.\n\nThe word "crusty" has a double meaning here:\n1. In a literal sense, a pizza has a crust, which is the outer layer of the bread.\n2. In an idiomatic sense, "feeling crusty" is an expression that means being irritable, grumpy, or in a bad mood.\n\nThe humor comes from the unexpected twist of using a word that is closely related to the physical properties of a pizza (its crust) to describe its emotional state (being in a bad mood). The joke requires a quick mental connection between the two mea

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)', 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to emotions or a personal issue. However, the punchline "Because it was feeling a little crusty!" subverts this expectation by using a wordplay.\n\nThe word "crusty" has a double meaning here:\n1. In a literal sense, a pizza has a crust, which is the outer layer of the bread.\n2. In an idiomatic sense, "feeling crusty" is an expression that means being irritable, grumpy, or in a bad mood.\n\nThe humor comes from the unexpected twist of using a word that is closely related to the physical properties of a pizza (its crust) to describe its emotional state (being in a bad mood). The joke requires a quick mental connection 

In [ ]:
# this is state what happenend till now in graph or state and what is stored 
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)', 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to emotions or a personal issue. However, the punchline "Because it was feeling a little crusty!" subverts this expectation by using a wordplay.\n\nThe word "crusty" has a double meaning here:\n1. In a literal sense, a pizza has a crust, which is the outer layer of the bread.\n2. In an idiomatic sense, "feeling crusty" is an expression that means being irritable, grumpy, or in a bad mood.\n\nThe humor comes from the unexpected twist of using a word that is closely related to the physical properties of a pizza (its crust) to describe its emotional state (being in a bad mood). The joke requires a quick mental connection

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment!',
 'explanation': 'A clever play on words. This joke is funny because it uses a common phrase associated with relationships, "tangled up in a lifelong commitment," and gives it a literal twist. In this case, the spaghetti is afraid of getting married because it\'s worried about getting "tangled up" - a phrase that typically refers to the complexities and potential problems that can arise in a long-term relationship.\n\nHowever, spaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. So, in this joke, the phrase "tangled up" takes on a double meaning. The spaghetti is making a pun on the phrase, using its own physical properties to create a humorous connection between the idea of commitment and the literal possibility of getting tangled.\n\nThe joke relies on this wordplay to create humor, and the unex

Time Travel

In [ ]:
# here by checkpoint_id we can go to our chat history and continue chat 
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)', 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to emotions or a personal issue. However, the punchline "Because it was feeling a little crusty!" subverts this expectation by using a wordplay.\n\nThe word "crusty" has a double meaning here:\n1. In a literal sense, a pizza has a crust, which is the outer layer of the bread.\n2. In an idiomatic sense, "feeling crusty" is an expression that means being irritable, grumpy, or in a bad mood.\n\nThe humor comes from the unexpected twist of using a word that is closely related to the physical properties of a pizza (its crust) to describe its emotional state (being in a bad mood). The joke requires a quick mental connection

Updating State

In [15]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1842a3-5731-6f82-8000-fcc0aa5babb3'}}

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1842a3-5731-6f82-8000-fcc0aa5babb3'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-20T11:00:24.542604+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='27d8e294-be8e-25a3-e996-9376d56ab70d', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)', 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to em

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1842a3-5731-6f82-8000-fcc0aa5babb3'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-20T11:00:24.542604+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='27d8e294-be8e-25a3-e996-9376d56ab70d', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)', 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to em